In [1]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.signal import butter, filtfilt
import logging

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from mamba_ssm import Mamba

# ==========================================
# Global config (aligned with training)
# ==========================================
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

if not torch.cuda.is_available():
    raise RuntimeError("GPU required for visualization")
device = torch.device("cuda")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

TARGET_LEN = 6000
INPUT_CHANNEL = 3
FS = 100.0
PICK_TOLERANCE = 50          # 0.5 s tolerance
PICK_THRESHOLD_P = 0.02
PICK_THRESHOLD_S = 0.08

BATCH_SIZE = 32
OUTPUT_DIR = "./outputs"      # original model path
MODEL_PATH = os.path.join(OUTPUT_DIR, "model_best.pt")
REAL_NOISE_PATH = "./stead_real_noise_50000.npy"
HIST_DIR = "./outputs_histograms"   # new output folder for histograms
os.makedirs(HIST_DIR, exist_ok=True)

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

# Plot style
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.dpi'] = 300

# ==========================================
# Helper functions (identical to training)
# ==========================================
def butter_bandpass(lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return b, a

def bandpass_filter(data, lowcut, highcut, fs, order=4):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y = filtfilt(b, a, data, axis=-1)
    return y

def normalize_single_sample(x):
    x_norm = np.zeros_like(x, dtype=np.float32)
    for ch in range(INPUT_CHANNEL):
        mean = np.mean(x[:, ch])
        std = np.std(x[:, ch])
        if std < 1e-6:
            std = 1.0
        x_norm[:, ch] = (x[:, ch] - mean) / std
    return x_norm

def generate_triangle_label(length, peak_idx, width=40):
    label = np.zeros(length, dtype=np.float32)
    half_width = width // 2
    start = max(0, peak_idx - half_width)
    end = min(length, peak_idx + half_width)
    rise_len = peak_idx - start
    if rise_len > 0:
        label[start:peak_idx] = np.linspace(0, 1, rise_len)
    label[peak_idx] = 1.0
    fall_len = end - peak_idx
    if fall_len > 0:
        label[peak_idx+1:end] = np.linspace(1, 0, fall_len-1)
    return label

def generate_3channel_labels(y_2channel, width_p=20, width_s=40):
    n_samples, n_timesteps = y_2channel.shape[0], y_2channel.shape[1]
    y_3channel = np.zeros((n_samples, n_timesteps, 3), dtype=np.float32)
    for i in range(n_samples):
        p_idx = np.argmax(y_2channel[i, :, 0])
        s_idx = np.argmax(y_2channel[i, :, 1])
        y_3channel[i, :, 0] = generate_triangle_label(n_timesteps, p_idx, width=width_p)
        y_3channel[i, :, 1] = generate_triangle_label(n_timesteps, s_idx, width=width_s)
    y_3channel[..., 2] = 1.0 - y_3channel[..., 0] - y_3channel[..., 1]
    return np.clip(y_3channel, 0.0, 1.0)

# ==========================================
# Optimized model (num_heads=4, kernel=13, no redundant pre_norm)
# ==========================================
class StochasticDepthDropout(nn.Module):
    def __init__(self, drop_rate):
        super().__init__()
        self.drop_rate = drop_rate

    def forward(self, layer_output, residual):
        if self.training:
            keep_prob = 1 - self.drop_rate
            if torch.rand(1).item() < keep_prob:
                return residual + (layer_output / keep_prob)
            else:
                return residual
        else:
            return residual + layer_output

class DilatedConvBlock(nn.Module):
    """Two dilated conv layers with residual connection."""
    def __init__(self, in_channels, filters, kernel_size=13, dilation_rates=[1, 2], dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, filters, kernel_size,
                               dilation=dilation_rates[0], padding='same')
        self.norm1 = nn.LayerNorm(filters)
        self.act1 = nn.ReLU()
        self.conv2 = nn.Conv1d(filters, filters, kernel_size,
                               dilation=dilation_rates[1], padding='same')
        self.norm2 = nn.LayerNorm(filters)
        self.act2 = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.residual_conv = nn.Conv1d(in_channels, filters, 1) if in_channels != filters else nn.Identity()

    def forward(self, x):
        residual = self.residual_conv(x)
        x = self.conv1(x)
        x = self.norm1(x.permute(0, 2, 1)).permute(0, 2, 1)
        x = self.act1(x)
        x = self.conv2(x)
        x = self.norm2(x.permute(0, 2, 1)).permute(0, 2, 1)
        x = self.act2(x)
        x = self.dropout(x)
        return x + residual

class DualMambaAttentionBlock(nn.Module):
    """Multi-head attention + bidirectional Mamba (no redundant pre_norm)."""
    def __init__(self, channels, num_heads=4, dropout=0.1, drop_rate=0.0):
        super().__init__()
        self.channels = channels
        self.pre_conv = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=13, padding='same'),
            nn.ReLU()
        )
        self.mha_norm = nn.LayerNorm(channels)
        self.mha = nn.MultiheadAttention(
            embed_dim=channels,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.sdd_mha = StochasticDepthDropout(drop_rate)
        self.mamba_fwd = Mamba(d_model=channels, d_state=8, d_conv=2, expand=2)
        self.mamba_bwd = Mamba(d_model=channels, d_state=8, d_conv=2, expand=2)
        self.fusion = nn.Linear(2 * channels, channels)
        self.mamba_norm = nn.LayerNorm(channels)
        self.sdd_mamba = StochasticDepthDropout(drop_rate)

    def forward(self, x):
        residual_att = x
        x = self.pre_conv(x)
        x = x.permute(0, 2, 1)
        x_norm = self.mha_norm(x)
        att_out, _ = self.mha(x_norm, x_norm, x_norm)
        att_out = att_out.permute(0, 2, 1)
        x = self.sdd_mha(att_out, residual_att)

        residual_mamba = x
        temp_x = x.permute(0, 2, 1)
        out_fwd = self.mamba_fwd(temp_x)
        temp_x_rev = torch.flip(temp_x, dims=[1])
        out_bwd = self.mamba_bwd(temp_x_rev)
        out_bwd = torch.flip(out_bwd, dims=[1])
        out_combined = torch.cat([out_fwd, out_bwd], dim=-1)
        temp_x = self.fusion(out_combined)
        temp_x = self.mamba_norm(temp_x)
        temp_x = temp_x.permute(0, 2, 1)
        x = self.sdd_mamba(temp_x, residual_mamba)
        return x

class EncoderBlock(nn.Module):
    """Dilated conv block + downsampling."""
    def __init__(self, in_channels, out_channels, dilation_rates, dropout=0.1):
        super().__init__()
        self.dilated = DilatedConvBlock(in_channels, out_channels, 13, dilation_rates, dropout)
        self.down = nn.Conv1d(out_channels, out_channels, 13, stride=2, padding=6)

    def forward(self, x):
        skip = self.dilated(x)
        down = F.relu(self.down(skip))
        return skip, down

class DecoderBlock(nn.Module):
    """Upsampling + skip connection + two conv layers."""
    def __init__(self, in_channels, out_channels, skip_channels):
        super().__init__()
        self.up = nn.ConvTranspose1d(in_channels, out_channels, kernel_size=13,
                                     stride=2, padding=6, output_padding=1)
        conv_in = out_channels + skip_channels
        self.conv = nn.Sequential(
            nn.Conv1d(conv_in, out_channels, 13, padding=6),
            nn.ReLU(),
            nn.Conv1d(out_channels, out_channels, 13, padding=6),
            nn.ReLU()
        )

    def forward(self, x, skip):
        x = self.up(x)
        x = F.relu(x)
        x = torch.cat([x, skip], dim=1)
        x = self.conv(x)
        return x

class PhaseNet_Custom_DoubleMamba(nn.Module):
    """U-Net with dual Mamba bottleneck, num_heads=4."""
    def __init__(self, num_heads=4):
        super().__init__()
        self.enc1 = EncoderBlock(INPUT_CHANNEL, 16, [1, 2])
        self.enc2 = EncoderBlock(16, 32, [3, 4])
        self.enc3 = EncoderBlock(32, 64, [5, 6])
        self.enc4 = EncoderBlock(64, 128, [7, 8])

        self.dma1 = DualMambaAttentionBlock(128, num_heads=num_heads, dropout=0.1, drop_rate=0.0)
        self.dma2 = DualMambaAttentionBlock(128, num_heads=num_heads, dropout=0.1, drop_rate=0.1)

        self.dec4 = DecoderBlock(128, 128, 128)
        self.dec3 = DecoderBlock(128, 64, 64)
        self.dec2 = DecoderBlock(64, 32, 32)
        self.dec1 = DecoderBlock(32, 16, 16)

        self.final_conv = nn.Conv1d(16, 3, 1, padding='same')
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv1d, nn.ConvTranspose1d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = x.permute(0, 2, 1)            # (B, C, T)

        skip1, x = self.enc1(x)
        skip2, x = self.enc2(x)
        skip3, x = self.enc3(x)
        skip4, x = self.enc4(x)

        x = self.dma1(x)
        x = self.dma2(x)

        x = self.dec4(x, skip4)
        x = self.dec3(x, skip3)
        x = self.dec2(x, skip2)
        x = self.dec1(x, skip1)

        x = self.final_conv(x)
        x = x.permute(0, 2, 1)            # (B, T, C)
        x = torch.softmax(x, dim=-1)
        return x

# ==========================================
# Inference dataset (no augmentation, only normalization)
# ==========================================
class SimpleInferenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x_raw = self.X[idx]
        x_norm = normalize_single_sample(x_raw)
        return torch.tensor(x_norm, dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.float32)

# ==========================================
# Plot picking error distribution histograms
# ==========================================
def plot_picking_error_distribution(y_true, y_pred):
    logger.info("Generating picking error histograms...")
    earthquake_mask = np.max(y_true[..., 0], axis=1) > 0.5
    y_true_eq = y_true[earthquake_mask]
    y_pred_eq = y_pred[earthquake_mask]

    p_true_idx = np.argmax(y_true_eq[..., 0], axis=1)
    p_pred_idx = np.argmax(y_pred_eq[..., 0], axis=1)
    s_true_idx = np.argmax(y_true_eq[..., 1], axis=1)
    s_pred_idx = np.argmax(y_pred_eq[..., 1], axis=1)

    # Keep only valid picks (error ≤ 0.5 s)
    p_errors = (p_pred_idx - p_true_idx) / FS
    s_errors = (s_pred_idx - s_true_idx) / FS
    valid_p = np.abs(p_pred_idx - p_true_idx) <= PICK_TOLERANCE
    valid_s = np.abs(s_pred_idx - s_true_idx) <= PICK_TOLERANCE
    p_errors = p_errors[valid_p]
    s_errors = s_errors[valid_s]

    bins = np.linspace(-0.2, 0.2, 41)
    p_counts, _ = np.histogram(p_errors, bins=bins)
    s_counts, _ = np.histogram(s_errors, bins=bins)
    ymax = np.ceil(max(np.max(p_counts), np.max(s_counts)) / 1000) * 1000

    bar_color = '#a8d1e7'

    # P-wave histogram
    fig_p, ax_p = plt.subplots(figsize=(7, 6))
    ax_p.hist(p_errors, bins=bins, color=bar_color, edgecolor='white', alpha=0.8)
    mae_p = np.mean(np.abs(p_errors))
    std_p = np.std(p_errors)
    ax_p.text(0, 1, f'MAE = {mae_p:.3f} s\nσ = {std_p:.3f} s',
              transform=ax_p.transAxes, fontsize=12, va='top', ha='left',
              bbox=dict(facecolor='white', alpha=0.9, edgecolor='gray', boxstyle='round,pad=0.3'))
    ax_p.set_title('P-wave Picking Error Distribution', fontsize=14, fontweight='bold')
    ax_p.set_xlabel('Picking Error (s)', fontsize=12)
    ax_p.set_ylabel('Count', fontsize=12)
    ax_p.set_xlim([-0.2, 0.2])
    ax_p.set_ylim([0, ymax])
    ax_p.grid(alpha=0.2)
    plt.tight_layout()
    save_p = os.path.join(HIST_DIR, 'p_error_histogram.png')
    plt.savefig(save_p)
    plt.close(fig_p)
    logger.info(f"P-wave histogram saved to: {save_p}")

    # S-wave histogram
    fig_s, ax_s = plt.subplots(figsize=(7, 6))
    ax_s.hist(s_errors, bins=bins, color=bar_color, edgecolor='white', alpha=0.8)
    mae_s = np.mean(np.abs(s_errors))
    std_s = np.std(s_errors)
    ax_s.text(0, 1, f'MAE = {mae_s:.3f} s\nσ = {std_s:.3f} s',
              transform=ax_s.transAxes, fontsize=12, va='top', ha='left',
              bbox=dict(facecolor='white', alpha=0.9, edgecolor='gray', boxstyle='round,pad=0.3'))
    ax_s.set_title('S-wave Picking Error Distribution', fontsize=14, fontweight='bold')
    ax_s.set_xlabel('Picking Error (s)', fontsize=12)
    ax_s.set_ylabel('Count', fontsize=12)
    ax_s.set_xlim([-0.2, 0.2])
    ax_s.set_ylim([0, ymax])
    ax_s.grid(alpha=0.2)
    plt.tight_layout()
    save_s = os.path.join(HIST_DIR, 's_error_histogram.png')
    plt.savefig(save_s)
    plt.close(fig_s)
    logger.info(f"S-wave histogram saved to: {save_s}")

# ==========================================
# Main: load data, infer, plot histograms
# ==========================================
def main():
    logger.info("="*80)
    logger.info("  Optimized model (heads=4) – Picking error histograms")
    logger.info("="*80)

    # Load and split dataset (same 7:2:1 as training)
    logger.info("\nLoading dataset...")
    X_orig = np.load("chunk2_stead_X.npy").astype(np.float32)
    y_2ch = np.load("chunk2_stead_y.npy").astype(np.float32)
    all_real_noise = np.load(REAL_NOISE_PATH).astype(np.float32)

    y_3ch = generate_3channel_labels(y_2ch, width_p=20, width_s=40)

    indices = np.arange(len(X_orig))
    np.random.seed(SEED)
    np.random.shuffle(indices)
    n_total_eq = len(X_orig)
    n_train_eq = int(n_total_eq * 0.7)
    n_val_eq = int(n_total_eq * 0.2)

    test_eq_idx = indices[n_train_eq + n_val_eq:]
    X_test_eq = X_orig[test_eq_idx]
    y_test_eq = y_3ch[test_eq_idx]

    all_real_noise = all_real_noise[:50000]
    n_train_noise = int(50000 * 0.7)
    n_val_noise = int(50000 * 0.2)
    n_test_noise = 50000 - n_train_noise - n_val_noise
    noise_indices = np.arange(50000)
    np.random.shuffle(noise_indices)
    test_noise_idx = noise_indices[n_train_noise + n_val_noise:]
    X_test_noise = all_real_noise[test_noise_idx]
    y_test_noise = np.zeros((n_test_noise, TARGET_LEN, 3), dtype=np.float32)
    y_test_noise[..., 2] = 1.0

    X_test = np.concatenate([X_test_eq, X_test_noise], axis=0)
    y_test = np.concatenate([y_test_eq, y_test_noise], axis=0)
    logger.info(f"Test set: {len(X_test_eq)} events + {len(X_test_noise)} noise")

    # Load optimized model (num_heads=4)
    logger.info(f"\nLoading optimized model from: {MODEL_PATH}...")
    if not os.path.exists(MODEL_PATH):
        logger.error(f"Model file not found: {MODEL_PATH}")
        return
    model = PhaseNet_Custom_DoubleMamba(num_heads=4).to(device)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
    model.eval()
    logger.info("Model loaded successfully")

    # Inference
    logger.info("\nRunning inference on test set...")
    dataset = SimpleInferenceDataset(X_test, y_test)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
    all_pred = []
    with torch.no_grad():
        for x_batch, _ in tqdm(loader, desc="Inferencing"):
            pred = model(x_batch.to(device))
            all_pred.append(pred.cpu().numpy())
    all_pred = np.concatenate(all_pred, axis=0)
    logger.info("Inference complete")

    # Generate histograms
    plot_picking_error_distribution(y_test, all_pred)

    logger.info("\nHistograms saved to: " + HIST_DIR)

if __name__ == "__main__":
    main()

2026-06-05 00:03:39,198 - ================================================================================
2026-06-05 00:03:39,200 -   Optimized model (heads=4) – Picking error histograms
2026-06-05 00:03:39,201 - ================================================================================
2026-06-05 00:03:39,201 - 
Loading dataset...
2026-06-05 00:04:21,293 - Test set: 20000 events + 5000 noise
2026-06-05 00:04:21,294 - 
Loading optimized model from: ./outputs/model_best.pt...
2026-06-05 00:04:21,497 - Model loaded successfully
2026-06-05 00:04:21,498 - 
Running inference on test set...
Inferencing: 100%|████████████████████████████| 782/782 [00:09<00:00, 85.74it/s]
2026-06-05 00:04:30,964 - Inference complete
2026-06-05 00:04:30,965 - Generating picking error histograms...
2026-06-05 00:04:32,640 - P-wave histogram saved to: ./outputs_histograms/p_error_histogram.png
2026-06-05 00:04:32,860 - S-wave histogram saved to: ./outputs_histograms/s_error_histogram.png
2026-06-05 00:04:3